# Golden set ✨

Mostrando a aplicação do modelo em 5 exemplos

## 0. Configs

### 0.1 Imports

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
import numpy as np

from joblib import load

pd.set_option('display.max_columns', None)

### 0.2 Data

In [3]:
df = pd.read_parquet('../data/trusted/tabela_analitica.parquet', engine = 'pyarrow')

df.head()

,age,job,marital,education,faixa_etaria,housing,loan,contact,month,year,day_of_week,previous,poutcome,emp_var_rate,cons_price_idx,cons_conf_idx,euribor3m,nr_employed,y
0,56,housemaid,married,basic.4y,Entre 50 e 60 anos,no,no,telephone,05. may,2008,1. mon,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
1,57,services,married,high.school,Entre 50 e 60 anos,no,no,telephone,05. may,2008,1. mon,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
2,37,services,married,high.school,Entre 31 e 40 anos,yes,no,telephone,05. may,2008,1. mon,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
3,40,admin.,married,basic.6y,Entre 31 e 40 anos,no,no,telephone,05. may,2008,1. mon,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
4,56,services,married,high.school,Entre 50 e 60 anos,no,yes,telephone,05. may,2008,1. mon,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no


## 1. Seleção dos casos

Selecionando amostras de 2010 (teste) tentando maximizar a diversidade de categorias das colunas

In [8]:
colunas = [
    'month',
    'job',
    'education',
    'marital',
    'faixa_etaria',
    'housing',
    'loan',
    'day_of_week',
    'poutcome',
    'contact',
    'y'
]

df_temp = df.drop_duplicates(subset=colunas).copy()

selecionadas = []
valores_usados = {col: set() for col in colunas}

for _ in range(5):

    # Pontuação = quantos valores novos cada linha adicionaria
    scores = df_temp.apply(
        lambda row: sum(
            row[col] not in valores_usados[col]
            for col in colunas
        ),
        axis=1
    )

    # Entre as melhores opções, escolhe uma aleatoriamente
    melhores = scores[scores == scores.max()].index
    idx = melhores.to_series().sample(1, random_state=26).iloc[0]

    linha = df_temp.loc[idx]
    selecionadas.append(idx)

    # Registra as categorias que já apareceram
    for col in colunas:
        valores_usados[col].add(linha[col])

    df_temp = df_temp.drop(index=idx)

golden_set = df.loc[selecionadas]

golden_set

,age,job,marital,education,faixa_etaria,housing,loan,contact,month,year,day_of_week,previous,poutcome,emp_var_rate,cons_price_idx,cons_conf_idx,euribor3m,nr_employed,y
37464,30,admin.,married,university.degree,Entre 26 e 30 anos,yes,no,telephone,08. aug,2009,4. thu,0,nonexistent,-2.9,92.201,-31.4,0.873,5076.2,no
40279,25,self-employed,single,unknown,Até 25 anos,unknown,unknown,cellular,07. jul,2010,3. wed,2,failure,-1.7,94.215,-40.3,0.896,4991.6,yes
39124,73,retired,divorced,high.school,Mais de 60 anos,no,yes,cellular,12. dec,2009,2. tue,2,success,-3.0,92.713,-33.0,0.707,5023.5,yes
32663,45,technician,unknown,basic.6y,Entre 41 e 50 anos,no,no,cellular,05. may,2009,1. mon,1,failure,-1.8,92.893,-46.2,1.299,5099.1,no
9384,53,blue-collar,divorced,basic.4y,Entre 50 e 60 anos,no,no,telephone,06. jun,2008,5. fri,0,nonexistent,1.4,94.465,-41.8,4.967,5228.1,no


## 2. Predictions

In [10]:
# Carrega todos os artefatos necessários para inferência
bundle = load("../models/lints_bundle.joblib")

preprocessor = bundle["preprocessor"]
lints = bundle["bandit"]
reward_models = bundle["reward_models"]
arms = bundle["arms"]
context_features = bundle["context_features"]

golden_predictions = golden_set.copy()

X_golden = np.asarray(
    preprocessor.transform(golden_predictions[context_features]),
    dtype=np.float64
)

recommended_actions = np.asarray(
    lints.predict(contexts=X_golden)
).reshape(-1)

probabilidades = np.empty(len(golden_predictions))

for arm in arms:
    mask = recommended_actions == arm

    if mask.any():
        probabilidades[mask] = reward_models[arm].predict_proba(X_golden[mask])[:, 1]

golden_predictions["contact_recomendado"] = recommended_actions
golden_predictions["probabilidade_conversao"] = probabilidades

threshold = 0.50

golden_predictions["aceita_oferta"] = np.where(
    golden_predictions["probabilidade_conversao"] >= threshold,
    "yes",
    "no"
)

golden_predictions

,age,job,marital,education,faixa_etaria,housing,loan,contact,month,year,day_of_week,previous,poutcome,emp_var_rate,cons_price_idx,cons_conf_idx,euribor3m,nr_employed,y,contact_recomendado,probabilidade_conversao,aceita_oferta
37464,30,admin.,married,university.degree,Entre 26 e 30 anos,yes,no,telephone,08. aug,2009,4. thu,0,nonexistent,-2.9,92.201,-31.4,0.873,5076.2,no,cellular,0.327786,no
40279,25,self-employed,single,unknown,Até 25 anos,unknown,unknown,cellular,07. jul,2010,3. wed,2,failure,-1.7,94.215,-40.3,0.896,4991.6,yes,cellular,0.756418,yes
39124,73,retired,divorced,high.school,Mais de 60 anos,no,yes,cellular,12. dec,2009,2. tue,2,success,-3.0,92.713,-33.0,0.707,5023.5,yes,cellular,0.787896,yes
32663,45,technician,unknown,basic.6y,Entre 41 e 50 anos,no,no,cellular,05. may,2009,1. mon,1,failure,-1.8,92.893,-46.2,1.299,5099.1,no,cellular,0.091188,no
9384,53,blue-collar,divorced,basic.4y,Entre 50 e 60 anos,no,no,telephone,06. jun,2008,5. fri,0,nonexistent,1.4,94.465,-41.8,4.967,5228.1,no,cellular,0.063878,no


### Análise dos casos

| idx | Perfil | Contato feito | Contato recomendado | aceita oferta | predição | por que faz sentido
--- | --- | --- | --- | --- | --- | ---
| 1   | Admin com 30 anos, casado, com graduação, tem financiamento mas não tem empréstimo          | telefone      | celular             | não           | não      | A recomendação de **celular** é coerente com o padrão histórico de maior conversão desse canal. Apesar da mudança de canal, o modelo de recompensa estimou probabilidade inferior ao threshold de 0,5, resultando corretamente em **não conversão** para este caso.                             |
| 2   | Autônomo com 25 anos, solteiro, sem outros dados                                            | celular       | celular             | sim           | sim      | O modelo manteve o **celular**, que já era o canal observado e historicamente apresenta melhor conversão. A probabilidade estimada superou 0,5 e a previsão de **conversão** coincidiu com o resultado real, mostrando uma decisão consistente para esse perfil.                                |
| 3   | Aposentado com 73 anos, divorciado, com ensino médio, sem financiamento mas com empréstimo  | celular       | celular             | sim           | sim      | É um caso especialmente coerente com a análise exploratória: **aposentados e clientes de maior idade apresentaram maior propensão à contratação**, e o celular também apresentou melhor conversão histórica. O modelo recomendou esse canal e previu corretamente a **conversão**.              |
| 4   | Técnico com 46 anos, com ensino fundamental incompleto, sem financiamento nem empréstimo    | celular       | celular             | não           | não      | Embora o **celular** seja recomendado como o canal mais promissor, isso não implica necessariamente conversão. Para a combinação específica de características e contexto deste cliente, a probabilidade permaneceu abaixo de 0,5, e a previsão de **não conversão** coincidiu com o observado. |
| 5   | Blue-collar com 53 anos, divorciado, ensino fundamental 1, sem financiamento nem empréstimo | telefone      | celular             | não           | não      | O LinTS propôs trocar o canal observado de **telefone para celular**, coerentemente com a maior conversão histórica do celular. Ainda assim, o modelo de recompensa não encontrou evidência suficiente para estimar conversão acima do threshold, prevendo corretamente **não conversão**.      |


Observações:
- para uma análise mais aprofundada, seria interessante ver a evolução dos indicadores para entender se a situação econômica tem impacto na propensão de aceitar a oferta
- nos casos 1 e 5, como o contato histórico foi por telefone, mas o LinTS recomendou celular, não é possível afirmar a partir do histórico que a recomendação do braço estava “correta”: o resultado contrafactual de contatar esses clientes por celular não foi observado. O que é possível afirmar é que a predição de conversão estimada pelo reward model ficou abaixo de 0,5 e que a escolha do celular é compatível com a política aprendida pelo LinTS.